In [ ]:
!pip install evaluate

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import urllib.request
import os
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Khởi tạo thiết bị (GPU nếu có)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sử dụng thiết bị: {device}")

# Định nghĩa tên mô hình pre-trained
MODEL_NAME = "ProsusAI/finbert"

In [ ]:
# Đọc dữ liệu từ file JSON.
# Việc ánh xạ trực tiếp giúp tránh lỗi nhận diện nhầm các nhãn text thành giá trị NaN hoặc missing data.
df_train = pd.read_json('en_train.json')
df_test = pd.read_json('en_test.json')

# Kiểm tra dữ liệu đầu vào
print("Train head:")
print(df_train.head())
print("Test head:")
print(df_test.head())

# Ánh xạ nhãn văn bản sang dạng số nguyên (0, 1, 2)
label_mapping = {"positive": 0, "negative": 1, "neutral": 2}

def preprocess_dataframe(df):
    # Loại bỏ các dòng không có cột sentiment hoặc sentence hợp lệ trước khi map
    df = df.dropna(subset=['sentence', 'sentiment'])
    df['label'] = df['sentiment'].map(label_mapping)

    # Loại bỏ các nhãn không nằm trong mapping (nếu có)
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    return df

# Áp dụng tiền xử lý cho cả 2 tập dữ liệu độc lập
df_train = preprocess_dataframe(df_train)
df_test = preprocess_dataframe(df_test)

# Chuyển đổi pandas DataFrame sang Hugging Face Dataset
train_dataset = Dataset.from_pandas(df_train[['sentence', 'label']]).remove_columns(['__index_level_0__'] if '__index_level_0__' in df_train.columns else [])
eval_dataset = Dataset.from_pandas(df_test[['sentence', 'label']]).remove_columns(['__index_level_0__'] if '__index_level_0__' in df_test.columns else [])

print(f"Số lượng mẫu Train: {len(train_dataset)}")
print(f"Số lượng mẫu Eval: {len(eval_dataset)}")

In [ ]:
# Load tokenizer của FinBERT
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    # Padding và truncation theo max_length chuẩn của BERT (512)
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=256)

# Áp dụng tokenize lên toàn bộ dataset
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# Data collator hỗ trợ padding động trong quá trình tạo batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Load metric từ thư viện evaluate
# Load từng metric độc lập
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Tính accuracy (không dùng tham số average)
    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )["accuracy"]

    # Tính f1, precision, recall (cần tham số average="macro")
    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )["f1"]

    precision = precision_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )["precision"]

    recall = recall_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )["recall"]

    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
# Tính toán class weights từ tập train để giải quyết chênh lệch phân phối
labels_train = np.array(train_dataset['label'])
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels_train), y=labels_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

# Khởi tạo Custom Trainer để áp dụng class weights vào CrossEntropyLoss
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# 1. Tải file trọng số gốc
url = "https://huggingface.co/ProsusAI/finbert/resolve/main/pytorch_model.bin"
if not os.path.exists("finbert_pytorch_model.bin"):
    urllib.request.urlretrieve(url, "finbert_pytorch_model.bin")

# 2. Đọc state_dict và đổi tên key từ gamma/beta sang weight/bias
state_dict = torch.load("finbert_pytorch_model.bin", map_location="cpu", weights_only=True)

for key in list(state_dict.keys()):
    if "LayerNorm.gamma" in key:
        state_dict[key.replace("LayerNorm.gamma", "LayerNorm.weight")] = state_dict.pop(key)
    elif "LayerNorm.beta" in key:
        state_dict[key.replace("LayerNorm.beta", "LayerNorm.bias")] = state_dict.pop(key)

# 3. Chỉ tải cấu hình (Config) của mô hình thay vì tải toàn bộ trọng số lỗi
config = AutoConfig.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3,
    id2label={0: "positive", 1: "negative", 2: "neutral"},
    label2id={"positive": 0, "negative": 1, "neutral": 2}
)

# 4. Khởi tạo mô hình trống dựa trên cấu hình
model = AutoModelForSequenceClassification.from_config(config)

# 5. Nạp trọng số đã sửa vào mô hình
model.load_state_dict(state_dict, strict=False)

training_args = TrainingArguments(
    output_dir="./finbert_finetuned_results",
    learning_rate=1e-5,               # Giảm LR
    warmup_ratio=0.1,                 # Thêm warmup
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    fp16=True,
    num_train_epochs=10,
    weight_decay=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    seed=42
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # THÊM Ở ĐÂY: Dừng sớm nếu val loss không cải thiện sau 2 epoch
)

In [ ]:
# Bắt đầu quá trình fine-tuning
trainer.train()

# Đánh giá trên tập test sau khi train xong
eval_results = trainer.evaluate()

# Lấy logit dự đoán và nhãn thực tế từ tập Validation/Evaluation
predictions, labels, metrics = trainer.predict(tokenized_eval)

# Chuyển đổi logit thành nhãn dự đoán (lấy class có xác suất cao nhất)
preds = np.argmax(predictions, axis=1)

# Tính toán Confusion Matrix
cm = confusion_matrix(labels, preds)

# Lấy danh sách tên nhãn tương ứng với mapping (0: positive, 1: negative, 2: neutral)
class_names = ["Positive", "Negative", "Neutral"]

# Trực quan hóa bằng biểu đồ Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Labels', fontsize=12)
plt.ylabel('True Labels', fontsize=12)
plt.title('Confusion Matrix on Evaluation Dataset', fontsize=14)
plt.show()

print("\nKết quả đánh giá trên tập Evaluation:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")